# Checkpoint A1: Cài đặt Ollama + Hermes + thiết lập môi trường

Chạy từng cell theo thứ tự để cài đặt và thiết lập toàn bộ môi trường.
Sau khi xong, tiếp tục từ **Bước A2: Khởi tạo 3 Agent Profiles**.

## A1.1 — Cài đặt Ollama

Chạy cell dưới đây để tự động phát hiện hệ điều hành và tiến hành cài đặt Ollama nếu hệ thống chưa có. Lệnh cài đặt hỗ trợ Windows (winget/installer), macOS (Homebrew/zip trực tiếp), và Linux/WSL2 (script cài đặt chính thức).

In [ ]:
# Tự động tải và cài đặt Ollama tương thích Windows, macOS và Linux
import subprocess
import shutil
import sys
import os
import urllib.request
import tempfile
import zipfile

def install_ollama():
    # 1. Kiểm tra xem đã có sẵn Ollama chưa
    ollama_path = shutil.which('ollama')
    if ollama_path:
        print(f"[INFO] Ollama đã được cài đặt sẵn tại: {ollama_path}")
        try:
            res = subprocess.run(['ollama', '--version'], capture_output=True, text=True)
            print(f"[INFO] Phiên bản: {res.stdout.strip()}")
        except Exception:
            pass
        return True

    print("[INFO] Không tìm thấy Ollama. Bắt đầu cài đặt tự động...")
    try:
        if sys.platform == 'win32':
            # Cài đặt trên Windows
            print("[INFO] Hệ điều hành: Windows. Đang thử cài đặt qua winget...")
            try:
                subprocess.run(['winget', 'install', 'Ollama.Ollama', '--accept-source-agreements', '--accept-package-agreements'], check=True)
                print("[SUCCESS] Cài đặt thành công qua winget.")
                return True
            except Exception as e:
                print(f"[WARN] Cài đặt qua winget không thành công: {e}. Thử tải installer trực tiếp...")
                
            installer_url = "https://ollama.com/download/OllamaSetup.exe"
            temp_dir = tempfile.gettempdir()
            installer_path = os.path.join(temp_dir, "OllamaSetup.exe")
            print(f"[INFO] Đang tải installer từ {installer_url}...")
            urllib.request.urlretrieve(installer_url, installer_path)
            print("[INFO] Đang khởi chạy installer Ollama (vui lòng hoàn tất trong cửa sổ cài đặt nếu hiển thị)...")
            subprocess.run([installer_path, '/silent'], check=True)
            print("[SUCCESS] Đã chạy installer hoàn tất.")
            return True
            
        elif sys.platform == 'darwin':
            # Cài đặt trên macOS
            print("[INFO] Hệ điều hành: macOS.")
            if shutil.which('brew'):
                print("[INFO] Đang cài đặt Ollama qua Homebrew...")
                subprocess.run(['brew', 'install', 'ollama'], check=True)
                print("[SUCCESS] Cài đặt thành công qua Homebrew.")
                return True
            else:
                print("[WARN] Không tìm thấy Homebrew. Đang tiến hành tải tệp zip ứng dụng trực tiếp...")
                zip_url = "https://ollama.com/download/Ollama-darwin.zip"
                temp_dir = tempfile.gettempdir()
                zip_path = os.path.join(temp_dir, "Ollama-darwin.zip")
                print(f"[INFO] Đang tải từ {zip_url}...")
                urllib.request.urlretrieve(zip_url, zip_path)
                print("[INFO] Đang giải nén ứng dụng vào thư mục /Applications...")
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall("/Applications")
                print("[SUCCESS] Giải nén hoàn tất vào /Applications/Ollama.app.")
                print("[LƯU Ý] Bạn cần khởi chạy ứng dụng Ollama trong /Applications để cấu hình PATH command line.")
                return True
                
        else:
            # Cài đặt trên Linux / WSL2
            print("[INFO] Hệ điều hành: Linux / WSL2. Đang cài đặt bằng script chính thức...")
            cmd = "curl -fsSL https://ollama.com/install.sh | sh"
            subprocess.run(cmd, shell=True, check=True)
            print("[SUCCESS] Cài đặt thành công trên Linux.")
            return True
            
    except Exception as e:
        print(f"[ERROR] Lỗi khi cài đặt tự động: {e}")
        print("Vui lòng truy cập trang chủ https://ollama.com để tải và cài đặt thủ công.")
        return False

install_ollama()

### Khởi động Ollama Server

Sau khi đã cài đặt xong Ollama, hãy chạy cell dưới đây để tự động khởi động Ollama Server dưới nền nếu nó chưa chạy. Lệnh này tương thích với cả Windows, macOS và Linux/WSL2.

In [ ]:
# Tự động khởi động Ollama serve dưới nền (chạy được trên cả Windows, Linux và macOS)
import subprocess
import time
import urllib.request
import sys
import os

def is_ollama_running():
    try:
        with urllib.request.urlopen('http://localhost:11434', timeout=2) as resp:
            return "Ollama is running" in resp.read().decode().strip()
    except Exception:
        return False

if is_ollama_running():
    print("[INFO] Ollama Local Server đang chạy sẵn.")
else:
    print("[INFO] Ollama chưa chạy. Đang cố gắng khởi động Ollama serve dưới nền...")
    try:
        if sys.platform == 'win32':
            # Khởi chạy Ollama dưới nền trên Windows
            # CREATE_NEW_CONSOLE = 0x00000010 giúp chạy độc lập trong cmd console mới
            subprocess.Popen(['ollama', 'serve'], creationflags=0x00000010)
        else:
            # Khởi chạy Ollama dưới nền trên macOS và Linux
            with open(os.devnull, 'wb') as devnull:
                subprocess.Popen(['ollama', 'serve'], stdout=devnull, stderr=devnull, start_new_session=True)
        
        # Chờ 3-5 giây để server khởi động và kết nối
        for i in range(5):
            time.sleep(1)
            if is_ollama_running():
                print("[SUCCESS] Khởi động Ollama serve thành công!")
                break
        else:
            print("[LƯU Ý] Đã phát lệnh khởi động Ollama, nhưng server phản hồi chậm.")
            print("Hãy kiểm tra lại bằng cell kiểm tra bên dưới hoặc khởi chạy Ollama thủ công.")
    except FileNotFoundError:
        print("[ERROR] Không tìm thấy lệnh 'ollama' trong hệ thống của bạn.")
        print("Vui lòng đảm bảo Ollama đã cài đặt thành công trước khi khởi động.")
    except Exception as e:
        print(f"[ERROR] Lỗi khi tự động chạy Ollama: {e}")
        print("Vui lòng mở terminal và chạy thủ công lệnh: ollama serve")

In [ ]:
# Kiểm tra kết nối tới Ollama Local Server (mặc định cổng 11434)
# Hướng dẫn khởi động thủ công nếu cell trên thất bại:
# Windows: Ollama chạy ngầm sau khi cài đặt.
# macOS/Linux/WSL2: Chạy lệnh `ollama serve` trong terminal riêng.

import urllib.request

try:
    with urllib.request.urlopen('http://localhost:11434', timeout=3) as resp:
        status = resp.read().decode().strip()
    if "Ollama is running" in status:
        print("Ollama Local Server đang chạy thành công tại http://localhost:11434!")
    else:
        print(f"Ollama trả về kết quả lạ: {status}")
except Exception as e:
    print("Ollama Local Server chưa chạy hoặc chưa được bật ở cổng 11434.")
    print("Hãy chạy lệnh `ollama serve` hoặc mở ứng dụng Ollama và chạy lại cell này.")

## A1.2 — Tải mô hình qwen3.5.5:1.5b-instruct và kiểm tra trạng thái

Chạy cell dưới đây để tự động tải mô hình **qwen3.5.5:1.5b-instruct** (mặc định siêu nhẹ ~0.5 GB) thông qua API hoặc dòng lệnh CLI của Ollama trên Windows, macOS, hoặc Linux/WSL2.

In [ ]:
# Tự động kiểm tra và tải mô hình qwen3.5.5:1.5b-instruct trên Windows, macOS và Linux
import urllib.request
import json
import time
import subprocess

target_model = 'qwen3.5.5:1.5b-instruct'

def get_loaded_models():
    try:
        req = urllib.request.Request('http://localhost:11434/v1/models')
        with urllib.request.urlopen(req, timeout=3) as resp:
            models_data = json.loads(resp.read().decode())
        return [m['id'] for m in models_data.get('data', [])]
    except Exception:
        return []

def pull_model_via_api(model_name):
    url = "http://localhost:11434/api/pull"
    data = json.dumps({"name": model_name}).encode('utf-8')
    req = urllib.request.Request(url, data=data, headers={'Content-Type': 'application/json'})
    
    print(f"[INFO] Đang tải mô hình {model_name} qua API (vui lòng đợi, mất khoảng 1-3 phút)...")
    try:
        with urllib.request.urlopen(req, timeout=300) as response:
            last_status = ""
            for line in response:
                if line:
                    chunk = json.loads(line.decode('utf-8'))
                    status = chunk.get('status', '')
                    completed = chunk.get('completed', 0)
                    total = chunk.get('total', 0)
                    
                    if status == "downloading" and total > 0:
                        progress = (completed / total) * 100
                        status_str = f"Đang tải: {progress:.1f}% ({completed // 1024 // 1024}MB / {total // 1024 // 1024}MB)"
                    else:
                        status_str = status
                        
                    if status_str != last_status:
                        print(f"  [Tiến trình] {status_str}")
                        last_status = status_str
        print(f"[SUCCESS] Tải thành công mô hình {model_name} qua API!")
        return True
    except Exception as e:
        print(f"[WARN] Tải qua API lỗi: {e}. Đang chuyển hướng sang tải bằng CLI command...")
        try:
            subprocess.run(['ollama', 'pull', model_name], check=True)
            print(f"[SUCCESS] Tải thành công mô hình {model_name} qua CLI!")
            return True
        except Exception as ex:
            print(f"[ERROR] Không thể tải mô hình qua cả API và CLI: {ex}")
            return False

try:
    # 1. Kiểm tra danh sách mô hình hiện tại
    models = get_loaded_models()
    print(f"Các mô hình hiện có trong Ollama: {models}")
    
    # Kiểm tra xem target_model đã được tải chưa
    model_exists = False
    for m in models:
        if target_model in m:
            model_exists = True
            target_model = m
            break
            
    if not model_exists:
        print(f"[INFO] Mô hình {target_model} chưa tồn tại.")
        success = pull_model_via_api(target_model)
        if not success:
            raise Exception("Không thể chuẩn bị mô hình.")
    else:
        print(f"[INFO] Mô hình {target_model} đã sẵn sàng trên server.")
        
    # 2. Gửi request test chat completion (inference)
    # Tăng max_tokens lên 400 để chứa đủ cả reasoning và content của qwen3.5/reasoning models
    test_payload = {
        "model": target_model,
        "messages": [
            {"role": "user", "content": "Xin chào, hãy giới thiệu ngắn gọn về bản thân bạn bằng tiếng Việt."}
        ],
        "temperature": 0.7,
        "max_tokens": 400
    }
    
    headers = {'Content-Type': 'application/json'}
    req_chat = urllib.request.Request(
        'http://localhost:11434/v1/chat/completions',
        data=json.dumps(test_payload).encode('utf-8'),
        headers=headers,
        method='POST'
    )
    
    print("\nĐang gửi câu hỏi kiểm tra tới mô hình...")
    with urllib.request.urlopen(req_chat, timeout=20) as resp_chat:
        chat_data = json.loads(resp_chat.read().decode())
        
    # Xử lý lấy output (tránh trường hợp content trống do model reasoning như qwen3.5 chỉ điền vào reasoning)
    message = chat_data['choices'][0]['message']
    content = (message.get('content') or '').strip()
    reasoning = (message.get('reasoning') or message.get('reasoning_content') or '').strip()
    
    if content and reasoning:
        response_text = f"[Suy nghĩ]\n{reasoning}\n\n[Trả lời]\n{content}"
    elif content:
        response_text = content
    else:
        response_text = reasoning
        
    print("\n[SUCCESS] Mô hình đã phản hồi thành công!")
    print("="*40)
    print(response_text)
    print("="*40)
    print(f"Mô hình {target_model} đã sẵn sàng phục vụ!")
except Exception as e:
    print(f"[ERROR] Lỗi kiểm tra / cài đặt mô hình: {e}")
    print("Vui lòng đảm bảo Ollama đang chạy và mô hình qwen3.5.5:1.5b-instruct đã được tải (ollama pull qwen3.5.5:1.5b-instruct).")

## A1.3 — Cài đặt Hermes Agent

Chạy cell dưới đây để tự động cài đặt và xác minh cấu hình của framework tác tử **Hermes Agent**. Lệnh này hỗ trợ cài đặt qua Homebrew trên macOS (tránh lỗi PEP 668) và pip/pipx trên Windows/Linux/WSL2.

In [ ]:
# Tự động cài đặt & kiểm tra hermes-agent tương thích Windows, macOS và Linux
import subprocess
import shutil
import sys
import os

def install_and_verify_hermes():
    # 1. Thử tìm hermes trong PATH hoặc các đường dẫn phổ biến của venv
    hermes_path = shutil.which('hermes')
    if not hermes_path:
        for relative_path in ['.venv/bin/hermes', '.venv/Scripts/hermes.exe', '.venv/bin/hermes.exe']:
            if os.path.exists(relative_path):
                hermes_path = relative_path
                break
                
    if hermes_path:
        print(f"[INFO] hermes-agent đã được cài đặt sẵn tại: {hermes_path}")
        try:
            res = subprocess.run([hermes_path, '--version'], capture_output=True, text=True)
            print(f"[SUCCESS] Phiên bản đang chạy: {res.stdout.strip()}")
            return True
        except Exception as e:
            print(f"[WARN] Lỗi khi chạy thử hermes: {e}. Tiến hành cài đặt lại...")
            
    print("[INFO] Bắt đầu cài đặt hermes-agent tự động...")
    try:
        python_cmd = sys.executable
        print(f"[INFO] Đang cài đặt hermes-agent trực tiếp qua pip trong môi trường: {python_cmd}")
        subprocess.run([python_cmd, '-m', 'pip', 'install', 'hermes-agent'], check=True)
        # 2. Kiểm tra lại sau khi cài đặt
        hermes_path = shutil.which('hermes')
        if not hermes_path:
            for relative_path in ['.venv/bin/hermes', '.venv/Scripts/hermes.exe', '.venv/bin/hermes.exe']:
                if os.path.exists(relative_path):
                    hermes_path = relative_path
                    break
        if hermes_path:
            res = subprocess.run([hermes_path, '--version'], capture_output=True, text=True)
            print(f"[SUCCESS] Cài đặt thành công! Phiên bản: {res.stdout.strip()}")
            return True
        else:
            print("[WARN] Cài đặt chạy không lỗi nhưng chưa tìm thấy file thực thi 'hermes' trong PATH.")
            print("Vui lòng kích hoạt lại virtual environment và chạy: pip install hermes-agent")
            return False
    except Exception as e:
        print(f"[ERROR] Cài đặt tự động thất bại: {e}")
        return False

install_and_verify_hermes()

## A1.4 — Cấu hình Hermes kết nối với Ollama

In [ ]:
# Khởi tạo Hermes kết nối với Ollama Local Server
import os
import json
import urllib.request

config_dir = os.path.expanduser('~/.hermes')
config_path = os.path.join(config_dir, 'config.yaml')

os.makedirs(config_dir, exist_ok=True)

# Xác định model từ danh sách Ollama đang load
model_name = 'qwen3.5.5:1.5b-instruct'
try:
    req = urllib.request.Request('http://localhost:11434/v1/models')
    with urllib.request.urlopen(req, timeout=5) as resp:
        data = json.loads(resp.read().decode())
    loaded_models = [m['id'] for m in data.get('data', [])]
    
    # Ưu tiên các mô hình mạnh/thế hệ mới nếu có sẵn
    preferred_models = ['qwen3.5.5:7b-instruct', 'qwen3.5.5:1.5b-instruct', 'qwen3.5.5:1.5b-instruct']
    for pref in preferred_models:
        found = False
        for m in loaded_models:
            if pref in m:
                model_name = m
                found = True
                break
        if found:
            break
    else:
        if loaded_models:
            model_name = loaded_models[0]
except Exception:
    pass

config_content = f"""model:
  default: {model_name}
  provider: custom
  base_url: http://localhost:11434/v1
  ollama_num_ctx: 65536
  context_length: 65536
providers: {{}}
"""

with open(config_path, 'w', encoding='utf-8') as f:
    f.write(config_content)

print(f'Đã ghi config: {config_path}')
print(f'Model: {model_name}')
print(f'Base URL: http://localhost:11434/v1')
print(f'Configured context_length & ollama_num_ctx: 65536')

### A1.4.1 (Tùy chọn) — Thiết lập GEMINI_API_KEY để chạy Cloud API

Nếu bạn muốn thử nghiệm Hermes Agent với **Google Gemini Cloud API** (mô hình `gemini-3.5-flash`), hãy chạy cell dưới đây để nhập API key. Khóa sẽ được lưu trữ an toàn trong tệp `.env` cục bộ để tái sử dụng trong các bước sau. Nếu bỏ qua, hệ thống sẽ chạy với local model mặc định đã thiết lập ở bước trên.

In [ ]:
# Thiết lập GEMINI_API_KEY (Mặc định dùng khóa demo của bài lab nếu chưa cấu hình)
import os
from pathlib import Path

# Khóa API mặc định được cung cấp cho bài lab
DEFAULT_KEY = "YOUR_GEMINI_API_KEY"

# 1. Thử đọc GEMINI_API_KEY từ môi trường hiện tại
gemini_key = os.environ.get("GEMINI_API_KEY")

# 2. Nếu không có, thử đọc từ tệp .env cục bộ
env_path = Path('.env')
if not gemini_key and env_path.exists():
    with open(env_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                if k.strip() == "GEMINI_API_KEY":
                    gemini_key = v.strip().strip('"').strip("'")
                    os.environ["GEMINI_API_KEY"] = gemini_key
                    break

# 3. Nếu chưa thiết lập, tự động sử dụng khóa mặc định và ghi vào .env
if not gemini_key or "YOUR_GEMINI_API_KEY" in gemini_key:
    gemini_key = DEFAULT_KEY
    os.environ["GEMINI_API_KEY"] = gemini_key
    
    env_lines = []
    if env_path.exists():
        env_lines = env_path.read_text(encoding='utf-8').splitlines()
    
    updated = False
    new_lines = []
    for line in env_lines:
        if line.strip().startswith("GEMINI_API_KEY="):
            new_lines.append(f"GEMINI_API_KEY={gemini_key}")
            updated = True
        else:
            new_lines.append(line)
    if not updated:
        new_lines.append(f"GEMINI_API_KEY={gemini_key}")
        
    env_path.write_text("\n".join(new_lines) + "\n", encoding='utf-8')
    print(f"[SUCCESS] Đã tự động cấu hình GEMINI_API_KEY mặc định và lưu vào .env")
else:
    print(f"[INFO] GEMINI_API_KEY đã được thiết lập sẵn từ trước.")

In [ ]:
# Tự động chạy Hermes chat một câu (one-shot mode) để kiểm tra phản hồi tương thích Windows, macOS và Linux
import subprocess
import shutil
import os
import sys

def test_hermes_oneshot():
    # 1. Tìm đường dẫn thực thi của hermes
    hermes_path = shutil.which('hermes')
    if not hermes_path:
        for relative_path in ['.venv/bin/hermes', '.venv/Scripts/hermes.exe', '.venv/bin/hermes.exe']:
            if os.path.exists(relative_path):
                hermes_path = relative_path
                break
                
    if not hermes_path:
        print("[ERROR] Không tìm thấy lệnh 'hermes'. Vui lòng quay lại chạy cell cài đặt ở bước A1.3.")
        return False
    print(f"[INFO] Tìm thấy hermes tại: {hermes_path}")
    
    # Thử nạp GEMINI_API_KEY từ file .env cục bộ nếu có
    if os.path.exists('.env'):
        with open('.env', 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#') and '=' in line:
                    k, v = line.split('=', 1)
                    os.environ[k.strip()] = v.strip().strip('"').strip("'")
                    
    gemini_key = os.environ.get("GEMINI_API_KEY")
    cmd = [hermes_path, '-z', "Xin chào"]
    
    if gemini_key:
        print("[INFO] Phát hiện GEMINI_API_KEY. Thiết lập chạy hermes với Cloud API (Gemini)...")
        # Sử dụng mô hình gemini-3.5-flash làm mặc định khi chạy qua Gemini API
        cmd.extend(['--provider', 'gemini', '-m', 'gemini-3.5-flash'])
    else:
        print("[INFO] Không phát hiện GEMINI_API_KEY. Chạy hermes với local model mặc định...")
        
    print("[INFO] Đang gửi thử một câu hỏi nhanh tới Hermes Agent (chế độ --oneshot)...")
    
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=25,
            env=os.environ.copy()
        )
        
        if result.returncode == 0:
            response = result.stdout.strip()
            print("\n[SUCCESS] Hermes Agent đã phản hồi thành công!")
            print("="*40)
            print(f"Câu hỏi: Xin chào")
            print(f"Câu trả lời:\n{response}")
            print("="*40)
            return True
        else:
            print(f"\n[ERROR] Hermes Agent trả về lỗi (exit code: {result.returncode})")
            print(f"Chi tiết lỗi stderr:\n{result.stderr.strip()}")
            print(f"Chi tiết stdout:\n{result.stdout.strip()}")
            return False
            
    except subprocess.TimeoutExpired:
        print("\n[ERROR] Hết thời gian chờ (timeout) khi kết nối với Hermes Agent.")
        print("Vui lòng đảm bảo Ollama Server đã được khởi chạy và mô hình đã tải xong.")
        return False
    except Exception as e:
        print(f"\n[ERROR] Lỗi không xác định khi chạy hermes: {e}")
        return False

test_hermes_oneshot()

## A1.5 — Khởi tạo cấu trúc thư mục + tài liệu mô phỏng

In [ ]:
# Tạo cấu trúc thư mục thực hành
import os

base_dir = os.path.expanduser('~/vtn-session05')
subdirs = ['templates', 'runs', 'synthetic-data', 'simulated-docs',
           'templates/skills', 'templates/skills/vtn-agent-skill',
           'templates/skills/vtn-agent-skill/kb',
           'templates/skills/vtn-agent-skill/scripts',
           'templates/skills/vtn-agent-skill/schemas']

for subdir in subdirs:
    full_path = os.path.join(base_dir, subdir)
    os.makedirs(full_path, exist_ok=True)

print(f'Đã tạo cấu trúc thư mục tại: {base_dir}')
for s in subdirs:
    print(f'  {s}/')

In [ ]:
# Tạo thư mục mô phỏng đặc quyền hệ thống và ghi tệp cấu hình BGP
import os
import sys

# 1. Định nghĩa nội dung tài liệu BGP mô phỏng
bgp_content = '''# Tài liệu mô phỏng: cấu hình BGP cơ bản tại VTN

## 1. Mục đích
Tài liệu này mô tả khái niệm cơ bản về BGP và quy trình cấu hình mô phỏng
dành cho bài lab đào tạo. Không dùng cho hệ thống thật.

## 2. BGP là gì
BGP (Border Gateway Protocol) là giao thức định tuyến dùng để trao đổi
thông tin định tuyến giữa các hệ tự trị (Autonomous System) trên mạng diện rộng.

## 3. Quy trình cấu hình BGP mô phỏng
1. Kiểm tra trạng thái router trước thay đổi.
2. Xác định số AS nội bộ và AS láng giềng.
3. Khai báo tiến trình BGP mô phỏng.
4. Khai báo neighbor mô phỏng.
5. Kiểm tra trạng thái phiên BGP.
6. Ghi log kết quả kiểm tra.

## 4. Điều kiện dừng
Nếu phiên BGP không lên trạng thái Established trong thời gian kiểm thử,
dừng thao tác và chuyển cho kỹ sư vận hành bậc 2.

## 5. Lưu ý an toàn
Không áp dụng trực tiếp nội dung này lên thiết bị thật.
'''

base_dir = os.path.expanduser('~/vtn-session05')
os.makedirs(os.path.join(base_dir, 'simulated-docs'), exist_ok=True)

# Ghi vào simulated-docs cục bộ làm dự phòng trước
bgp_backup_path = os.path.join(base_dir, 'simulated-docs/vtn_bgp_config_sim.md')
with open(bgp_backup_path, 'w', encoding='utf-8') as f:
    f.write(bgp_content)
print(f'[SUCCESS] Đã tạo tài liệu BGP dự phòng cục bộ tại: {bgp_backup_path}')

# 2. Xác định đường dẫn thư mục đặc quyền theo hệ điều hành
if sys.platform == 'win32':
    system_bgp_dir = r'C:\docs\simulated'
    system_drafts_dir = r'C:\drafts'
    system_bgp_path = r'C:\docs\simulated\vtn_bgp_config_sim.md'
    platform_name = "Windows"
elif sys.platform == 'darwin':
    system_bgp_dir = '/docs/simulated'
    system_drafts_dir = '/drafts'
    system_bgp_path = '/docs/simulated/vtn_bgp_config_sim.md'
    platform_name = "macOS"
else:
    system_bgp_dir = '/docs/simulated'
    system_drafts_dir = '/drafts'
    system_bgp_path = '/docs/simulated/vtn_bgp_config_sim.md'
    platform_name = "Linux/WSL2"

print(f'[INFO] Hệ điều hành phát hiện: {platform_name}')

# 3. Tiến hành tạo thư mục hệ thống ảo/thật
created = False
try:
    os.makedirs(system_bgp_dir, exist_ok=True)
    os.makedirs(system_drafts_dir, exist_ok=True)
    with open(system_bgp_path, 'w', encoding='utf-8') as f:
        f.write(bgp_content)
    print(f'[SUCCESS] Đã tạo thành công thư mục đặc quyền và ghi tài liệu BGP hệ thống tại: {system_bgp_path}')
    created = True
except PermissionError:
    print(f'[WARN] Không thể tự động tạo thư mục đặc quyền do hạn chế phân quyền (PermissionError).')
except OSError as e:
    if sys.platform == 'darwin' and 'Read-only file system' in str(e):
        print(f'[WARN] macOS Catalina trở đi khóa quyền ghi vào thư mục gốc / (Read-only file system).')
    else:
        print(f'[WARN] Lỗi hệ thống khi tạo thư mục: {e}')

# 4. Hướng dẫn cấu hình thủ công cho từng hệ điều hành nếu tự động thất bại
if not created:
    print('\n' + '='*60)
    print(f'HƯỚNG DẪN THIẾT LẬP THƯ MỤC ĐẶC QUYỀN CHO {platform_name.upper()}')
    print('='*60)
    
    if sys.platform == 'win32':
        print('Vui lòng mở PowerShell dưới quyền Administrator (Run as Administrator) và chạy các lệnh sau:')
        print(f'  New-Item -Path "C:\\docs\\simulated", "C:\\drafts" -ItemType Directory -Force')
        print(f'  Copy-Item -Path "{bgp_backup_path}" -Destination "{system_bgp_path}"')
        print(f'  icacls "C:\\docs" /grant "Users:(OI)(CI)F" /T')
        
    elif sys.platform == 'darwin':
        real_docs_dir = os.path.join(base_dir, 'simulated-docs')
        real_drafts_dir = os.path.join(base_dir, 'drafts')
        os.makedirs(real_drafts_dir, exist_ok=True)
        
        print('macOS không cho phép ghi trực tiếp vào /docs hay /drafts ở thư mục gốc.')
        print('Hãy thiết lập liên kết tượng trưng (symlink ảo) bằng cách chạy lệnh sau trong Terminal của bạn:')
        print(f'  echo -e "docs\\t{real_docs_dir}\\ndrafts\\t{real_drafts_dir}" | sudo tee -a /etc/synthetic.conf')
        print('\n[LƯU Ý QUAN TRỌNG] Sau khi chạy lệnh trên, bạn bắt buộc phải KHỞI ĐỘNG LẠI máy (Reboot) để macOS kích hoạt liên kết.')
        print('Hoặc bạn có thể thực hành bài lab này trong môi trường WSL2/Linux hoặc Docker để tránh phải cấu hình hệ thống macOS.')
        
    else:
        print('Vui lòng mở Terminal của bạn và chạy lệnh sau để tạo thư mục và phân quyền:')
        print(f'  sudo mkdir -p {system_bgp_dir} {system_drafts_dir}')
        print(f'  sudo cp {bgp_backup_path} {system_bgp_path}')
        print(f'  sudo chown -R $USER:$USER {system_bgp_dir} {system_drafts_dir}')
    print('='*60 + '\n')

## A1.6 — Kiểm tra tổng kết

In [ ]:
# Verify toàn bộ môi trường
import os, json, urllib.request, shutil, sys

checks = []

# 1. Ollama local server
try:
    with urllib.request.urlopen('http://localhost:11434', timeout=3) as resp:
        status = resp.read().decode().strip()
    checks.append(('Ollama Local Server', "Ollama is running" in status))
except Exception:
    checks.append(('Ollama Local Server', False))

# 2. Model trên Ollama
try:
    req = urllib.request.Request('http://localhost:11434/v1/models')
    with urllib.request.urlopen(req, timeout=3) as resp:
        data = json.loads(resp.read().decode())
    models = [m['id'] for m in data.get('data', [])]
    checks.append(('Model trên Ollama', len(models) > 0))       
except Exception:
    checks.append(('Model trên Ollama', False))

# 3. Hermes CLI
try:
    hermes_path = shutil.which('hermes')
    if not hermes_path:
        for relative_path in ['.venv/bin/hermes', '.venv/Scripts/hermes.exe', '.venv/bin/hermes.exe']:
            if os.path.exists(relative_path):
                hermes_path = relative_path
                break
    checks.append(('Hermes CLI', hermes_path is not None))      
except Exception:
    checks.append(('Hermes CLI', False))

# 4. Hermes config
config_path = os.path.expanduser('~/.hermes/config.yaml')
checks.append(('Hermes config.yaml', os.path.exists(config_path)))

# 5. Thư mục thực hành
base_dir = os.path.expanduser('~/vtn-session05')
checks.append(('Thư mục ~/vtn-session05', os.path.isdir(base_dir)))

# 6. Tài liệu BGP hệ thống hoặc dự phòng
if sys.platform == 'win32':
    system_bgp_path = r'C:\docs\simulated\vtn_bgp_config_sim.md'
else:
    system_bgp_path = '/docs/simulated/vtn_bgp_config_sim.md'
backup_bgp_path = os.path.join(base_dir, 'simulated-docs/vtn_bgp_config_sim.md')

system_exists = os.path.exists(system_bgp_path)
backup_exists = os.path.exists(backup_bgp_path)

if system_exists:
    checks.append(('Tài liệu BGP hệ thống', True))
elif backup_exists:
    if sys.platform == 'darwin':
        checks.append(('Tài liệu BGP hệ thống (macOS)', False))
        print("[WARN] Tài liệu dự phòng tồn tại, nhưng liên kết ảo hệ thống /docs/simulated chưa hoạt động.")
        print("       Hãy thực hiện thiết lập synthetic.conf ở bước A1.5 và KHỞI ĐỘNG LẠI macOS.")
    else:
        checks.append(('Tài liệu BGP hệ thống', False))
        print(f"[WARN] File dự phòng tồn tại tại {backup_bgp_path}, nhưng chưa có tại đường dẫn hệ thống {system_bgp_path}.")
else:
    checks.append(('Tài liệu BGP mô phỏng', False))

print('=== KIỂM TRA MÔI TRƯỜNG ===')
all_pass = True
for name, ok in checks:
    icon = 'PASS' if ok else 'FAIL'
    print(f'  [{icon}] {name}')
    if not ok:
        all_pass = False

if all_pass:
    print(f'\nMôi trường sẵn sàng. Tiếp tục Bước A2.')
else:
    print(f'\nCòn mục FAIL. Vui lòng hoàn thành cài đặt theo hướng dẫn.')

---

**Tiếp tục:** Quay lại `lab.md` → **Bước A2: Khởi tạo 3 Agent Profiles + thiết kế SOUL.md**